<a href="https://colab.research.google.com/github/Ridzzz0Alam/Data_Science/blob/main/DS_NaiveBayes_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

Load and explore the dataset

In [ ]:
csv_path = "fake_or_real_news.csv"
df_raw = pd.read_csv(csv_path)
print("Columns:", df_raw.columns.tolist())
df_raw.head()

If the first two columns are indeed 'text' and 'label', keep only them. If the column names look slightly different, adjust this slice accordingly.

In [ ]:
df = df_raw.iloc[:, :2]
df.columns = ["text", "label"]
print(df.info())
df.head()

Basic exploration

In [ ]:
print("Number of rows:", len(df))
print("\nClass distribution:")
print(df["label"].value_counts())

Quick look at some examples from each class

In [ ]:
print("\nSample FAKE article:\n", df[df["label"] == "FAKE"]["text"].iloc[0][:500], "...")
print("\nSample REAL article:\n", df[df["label"] == "REAL"]["text"].iloc[0][:500], "...")

2. Preprocess the data
    - Handle missing values
    - Ensure labels are clean

In [ ]:
df = df.dropna(subset=["text", "label"])
df["label"] = df["label"].str.strip()
print("Unique labels:", df["label"].unique())

3. Train/Test Split

In [ ]:
df = df[df['label'].isin(['FAKE', 'REAL'])]
X = df["text"].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

4. Build Naive Bayes Model (with TF-IDF)

In [ ]:
model = make_pipeline(
    TfidfVectorizer(
        stop_words='english',
        max_df=0.9,
        min_df=5
    ),
    MultinomialNB()
)

model.fit(X_train, y_train)

5. Evaluate Model Performance

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.4f}")
cm = confusion_matrix(y_test, y_pred, labels=["FAKE", "REAL"])
cm_df = pd.DataFrame(cm, index=["FAKE_true", "REAL_true"], columns=["FAKE_pred", "REAL_pred"])
print("\nConfusion Matrix:")
print(cm_df)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

6. Interpret Results
  Plot confusion matrix for visualization

In [ ]:
plt.figure(figsize=(4,3))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()